In [1]:
from datetime import datetime, timezone

import requests
import wikipedia
from dotenv import find_dotenv, load_dotenv
from langchain.agents import tool
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser
from langchain.agents.output_parsers.tools import ToolAgentAction
from langchain.chains.openai_functions.openapi import openapi_spec_to_openai_fn
from langchain.prompts import ChatPromptTemplate
from langchain.schema.agent import AgentFinish
from langchain.utilities.openapi import OpenAPISpec
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

In [2]:
load_dotenv(find_dotenv("../../creds/.env"), verbose=True);

In [ ]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="mistral-nemo:12b",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
    # num_ctx=16384,  # 40960 max
).bind(think=False)

In [4]:
class SearchInput(BaseModel):
    query: str = Field(description="Thing to search for")


@tool(args_schema=SearchInput)
def search(query: str) -> str:
    """Search for weather online"""

    return "42f"

In [5]:
search.args

{'query': {'description': 'Thing to search for',
  'title': 'Query',
  'type': 'string'}}

In [6]:
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")


@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> str:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1,
        "timezone": "GMT",
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.now(timezone.utc)
    time_list = [
        datetime.fromisoformat(time_str).replace(tzinfo=timezone.utc) for time_str in results["hourly"]["time"]
    ]
    temperature_list = results["hourly"]["temperature_2m"]

    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]

    return f"The current temperature is {current_temperature}°C"


get_current_temperature.invoke({"latitude": 13, "longitude": 14})

'The current temperature is 32.1°C'

In [7]:
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries"""

    page_titles = wikipedia.search(query)
    summaries = []

    for page_title in page_titles[:3]:
        try:
            wiki_page = wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            wikipedia.exceptions.PageError,
            wikipedia.exceptions.DisambiguationError,
        ):
            pass

    if not summaries:
        return "No good Wikipedia Search Result was found"

    return "\n\n".join(summaries)


print(search_wikipedia.invoke({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Retrieval-augmented generation
Summary: Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information. With RAG, LLMs do not respond to user queries until they refer to a specified set of documents. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this helps LLM-based chatbots access internal company data or generate responses based on authoritative sources.
RAG improves large language models (LLMs) by incor

In [8]:
text = """
{
  "openapi": "3.0.0",
  "info": {
    "version": "1.0.0",
    "title": "Swagger Petstore",
    "license": {
      "name": "MIT"
    }
  },
  "servers": [
    {
      "url": "http://petstore.swagger.io/v1"
    }
  ],
  "paths": {
    "/pets": {
      "get": {
        "summary": "List all pets",
        "operationId": "listPets",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "limit",
            "in": "query",
            "description": "How many items to return at one time (max 100)",
            "required": false,
            "schema": {
              "type": "integer",
              "maximum": 100,
              "format": "int32"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "A paged array of pets",
            "headers": {
              "x-next": {
                "description": "A link to the next page of responses",
                "schema": {
                  "type": "string"
                }
              }
            },
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pets"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      },
      "post": {
        "summary": "Create a pet",
        "operationId": "createPets",
        "tags": [
          "pets"
        ],
        "responses": {
          "201": {
            "description": "Null response"
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    },
    "/pets/{petId}": {
      "get": {
        "summary": "Info for a specific pet",
        "operationId": "showPetById",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "petId",
            "in": "path",
            "required": true,
            "description": "The id of the pet to retrieve",
            "schema": {
              "type": "string"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "Expected response to a valid request",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pet"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    }
  },
  "components": {
    "schemas": {
      "Pet": {
        "type": "object",
        "required": [
          "id",
          "name"
        ],
        "properties": {
          "id": {
            "type": "integer",
            "format": "int64"
          },
          "name": {
            "type": "string"
          },
          "tag": {
            "type": "string"
          }
        }
      },
      "Pets": {
        "type": "array",
        "maxItems": 100,
        "items": {
          "$ref": "#/components/schemas/Pet"
        }
      },
      "Error": {
        "type": "object",
        "required": [
          "code",
          "message"
        ],
        "properties": {
          "code": {
            "type": "integer",
            "format": "int32"
          },
          "message": {
            "type": "string"
          }
        }
      }
    }
  }
}
"""
spec = OpenAPISpec.from_text(text)
spec

Attempting to load an OpenAPI 3.0.0 spec.  This may result in degraded performance. Convert your OpenAPI spec to 3.1.* spec for better support.


OpenAPISpec(openapi='3.0.0', info=Info(title='Swagger Petstore', summary=None, description=None, termsOfService=None, contact=None, license=License(name='MIT', identifier=None, url=None), version='1.0.0'), jsonSchemaDialect=None, servers=[Server(url='http://petstore.swagger.io/v1', description=None, variables=None)], paths={'/pets': PathItem(ref=None, summary=None, description=None, get=Operation(tags=['pets'], summary='List all pets', description=None, externalDocs=None, operationId='listPets', parameters=[Parameter(description='How many items to return at one time (max 100)', required=False, deprecated=False, style=None, explode=None, param_schema=Schema(allOf=None, anyOf=None, oneOf=None, schema_not=None, schema_if=None, then=None, schema_else=None, dependentSchemas=None, prefixItems=None, items=None, contains=None, properties=None, patternProperties=None, additionalProperties=None, propertyNames=None, unevaluatedItems=None, unevaluatedProperties=None, type=<DataType.INTEGER: 'integ

In [9]:
pet_openai_functions, pet_callables = openapi_spec_to_openai_fn(spec)

In [10]:
llm_with_tools = llm.bind_tools(
    tools=pet_openai_functions,
)

In [ ]:
llm_with_tools.invoke("What are five pets names?").tool_calls

[{'name': 'listPets',
  'args': {},
  'id': '88ea9c79-dfdc-4e3c-9e45-7591abef0b17',
  'type': 'tool_call'}]

In [12]:
llm_with_tools.invoke("tell me about pet with id 42").tool_calls

[{'name': 'showPetById',
  'args': {'path_params': {'id': '42'}},
  'id': 'ed7274bd-a2ad-4428-93cf-e651335e7670',
  'type': 'tool_call'}]

### Routing

In [13]:
functions = [
    convert_to_openai_function(search_wikipedia),
    convert_to_openai_function(get_current_temperature),
]

llm_with_tools = llm.bind_tools(functions)

In [14]:
llm_with_tools.invoke("what is the weather is Kyiv right now?").tool_calls

[{'name': 'get_current_temperature',
  'args': {'latitude': 50.45, 'longitude': 30.52},
  'id': 'f902ac03-4927-4f94-b53d-3f063d44d82a',
  'type': 'tool_call'}]

In [15]:
prompt = ChatPromptTemplate.from_messages([("system", "You are helpful but sassy assistant"), ("user", "{input}")])
chain = prompt | llm_with_tools

In [16]:
chain.invoke({"input": "what is the weather is Kyiv right now?"})

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'mistral-nemo:12b', 'created_at': '2025-07-05T16:01:19.417934824Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1236862307, 'load_duration': 25896058, 'prompt_eval_count': 150, 'prompt_eval_duration': 20694342, 'eval_count': 36, 'eval_duration': 1184277075, 'model_name': 'mistral-nemo:12b'}, id='run--9b876787-555b-4c46-a413-2bc5516d404f-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 50.45, 'longitude': 30.52}, 'id': 'ac3acd39-4f96-4ee0-87f0-7d197859c440', 'type': 'tool_call'}], usage_metadata={'input_tokens': 150, 'output_tokens': 36, 'total_tokens': 186})

In [17]:
chain = prompt | llm_with_tools | OpenAIToolsAgentOutputParser()

In [18]:
result = chain.invoke({"input": "what is the weather is Kyiv right now?"})
result

[ToolAgentAction(tool='get_current_temperature', tool_input={'latitude': 50.45, 'longitude': 30.52}, log="\nInvoking: `get_current_temperature` with `{'latitude': 50.45, 'longitude': 30.52}`\n\n\n", message_log=[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'mistral-nemo:12b', 'created_at': '2025-07-05T16:01:20.675818553Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1224164280, 'load_duration': 26728688, 'prompt_eval_count': 150, 'prompt_eval_duration': 11581453, 'eval_count': 36, 'eval_duration': 1181827291, 'model_name': 'mistral-nemo:12b'}, id='run--aa334cc2-83d2-42ee-b2a6-b82429359e26-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 50.45, 'longitude': 30.52}, 'id': 'aeedb648-737c-4651-8e59-19bb4b197573', 'type': 'tool_call'}], usage_metadata={'input_tokens': 150, 'output_tokens': 36, 'total_tokens': 186})], tool_call_id='aeedb648-737c-4651-8e59-19bb4b197573')]

In [19]:
result = chain.invoke({"input": "hi"})
result

AgentFinish(return_values={'output': " Hi there! How can I assist you today? Let's keep it friendly and respectful, okay? 😊"}, log=" Hi there! How can I assist you today? Let's keep it friendly and respectful, okay? 😊")

In [ ]:
def route(results: list[ToolAgentAction]):
    if isinstance(results, AgentFinish):
        return results.return_values["output"]

    result = results[0]
    tools = {
        "search_wikipedia": search_wikipedia,
        "get_current_temperature": get_current_temperature,
    }

    return tools[result.tool].run(result.tool_input)

In [27]:
chain = prompt | llm_with_tools | OpenAIToolsAgentOutputParser() | route

In [28]:
result = chain.invoke({"input": "what is the weather is Kyiv right now?"})
result

'The current temperature is 24.3°C'

In [29]:
result = chain.invoke({"input": "what is langchain?"})
print(result)

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Intelligent agent
Summary: In artificial intelligence, an intelligent agent is an entity that perceives its environment, takes actions autonomously to achieve goals, and may improve its performance through machine learning or by acquiring knowledge. AI textbooks define artificial intelligence as the "study and design of intelligent agents," emphasizing that goal-directed behavior is central to intelligence.
A specialized subset of intelligent agents, agentic AI (also known as an AI agent or simply agent), expands this concept by proactively pursuing goals, making decisions, and taking actions over extended periods.
Intelligent agents ca

In [30]:
chain.invoke({"input": "hi"})

" Hi there! How can I assist you today? Let's keep it friendly and respectful, okay? 😊"